In [1]:
"""Module for implementing logistic regression using statsmodel.
This module standardizes and transforms the suicide study dataset, 
and applies logistic regression to analyze trends in suicide data 
from both 2023 and 2013-2022 periods."""

import pandas as pd
import numpy as np

import sys
from pathlib import Path
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

WORKSPACE_PATH = os.getenv("WORKSPACE_PATH")

# Add the parent directory to the system path
sys.path.append(str(WORKSPACE_PATH))

from src.config.config import DATA_DIR, RESULTS_DIR

Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\data
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\results
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\results\plots
Directory already exists: c:\Users\huber\OneDrive\Dokumenty\GitHub\ippan_suicide_study\results\tables


In [2]:
# ================================================================================
# Data reading
# ================================================================================

# Define the path to the CSV file containing latent class analysis results
csv_file_path = RESULTS_DIR / "poLCA" / "Group_AG.csv"

# Read the CSV file into a DataFrame, specifying that it should not use low memory mode
lca_classes = pd.read_csv(
    csv_file_path, delimiter=",", low_memory=False, index_col=None
)

# Define the path to the encoded dataset CSV file
csv_file_path = DATA_DIR / "encoded" / "encoded_final_set.csv"

# Read the encoded dataset into a DataFrame
df_encoded = pd.read_csv(csv_file_path, delimiter=",", low_memory=False, index_col=None)

# Merge the encoded dataset with the latent class analysis results on the "ID" column
df_encoded = df_encoded.merge(lca_classes, on="ID", how="left")

# Extract and sort the unique groups from the merged DataFrame
groups = sorted(list(set(df_encoded["Group_AG"])))

In [3]:
import statsmodels.api as sm

results_data = []

for group in groups:
    try:
        # Select Group from Group_AG
        group_data = df_encoded[df_encoded["Group_AG"] == group]

        group_data["Predicted_Class_Group_AG"] = group_data[
            "Predicted_Class_Group_AG"
        ].astype("category")

        # Selects columns
        X = pd.get_dummies(group_data["Predicted_Class_Group_AG"], drop_first=True)
        y = group_data["Fatal"]  # Zmienna zależna

        y = y.astype(int)
        X = X.astype(int)

        X = sm.add_constant(X)
        logreg_model = sm.Logit(y, X)
        result = logreg_model.fit(disp=0)

        # Wyodrębnij istotne informacje z modelu
        for param, coeff in result.params.items():
            results_data.append(
                {
                    "Group": group,
                    "Variable": param,
                    "Coefficient": coeff,
                    "P-value": result.pvalues[param],
                    "Standard Error": result.bse[param],
                    "Log-Likelihood": result.llf,
                    "AIC": result.aic,
                    "BIC": result.bic,
                }
            )
    except Exception as e:
        print(f"Error processing group {group}: {e}")

    # Tworzenie DataFrame z wynikami
    statsmodels_results_df = pd.DataFrame(results_data)

C:\Users\huber\AppData\Local\Temp\ipykernel_19340\3865596967.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group_data["Predicted_Class_Group_AG"] = group_data[
c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\huber\AppData\Local\Temp\ipykernel_19340\3865596967.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group_data["Predicted_Class_Group_AG"] = group_data[
C:

Error processing group 00_18_F: Singular matrix


C:\Users\huber\AppData\Local\Temp\ipykernel_19340\3865596967.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  group_data["Predicted_Class_Group_AG"] = group_data[


In [4]:
import statsmodels.api as sm
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    mean_squared_error,
)

# Initialize results list
results_data_statsmodels = []

for group in groups:
    try:
        # Select group data and make a copy
        group_data = df_encoded.loc[df_encoded["Group_AG"] == group].copy()

        # Ensure 'Predicted_Class_Group_AG' is categorical
        group_data.loc[:, "Predicted_Class_Group_AG"] = group_data[
            "Predicted_Class_Group_AG"
        ].astype("category")

        # Prepare features and target
        X = pd.get_dummies(group_data["Predicted_Class_Group_AG"], drop_first=True)
        X = X.loc[:, X.nunique() > 1]  # Drop constant columns

        # Ensure numeric data
        X = X.astype(int)
        y = group_data["Fatal"].astype(int)

        # Skip if insufficient samples or predictors
        if (
            X.shape[0] <= 1
            or X.shape[1] == 0
            or X.isnull().values.any()
            or y.isnull().any()
        ):
            print(f"Skipping group {group} due to insufficient data.")
            continue

        # Compute balanced class weights using sklearn
        class_weights = compute_class_weight(
            class_weight="balanced", classes=y.unique(), y=y
        )
        weight_mapping = dict(zip(y.unique(), class_weights))
        group_data["weights"] = y.map(weight_mapping)

        # Add constant for intercept
        X = sm.add_constant(X)

        # Fit logistic regression with statsmodels
        logit_model = sm.Logit(y, X, weights=group_data["weights"])
        result = logit_model.fit(disp=0)

        # Log-likelihood and metrics calculation
        log_likelihood = result.llf
        y_prob = result.predict(X)
        y_pred = (y_prob >= 0.5).astype(int)
        precision = precision_score(y, y_pred, zero_division=0)
        recall = recall_score(y, y_pred, zero_division=0)
        f1 = f1_score(y, y_pred, zero_division=0)
        accuracy = accuracy_score(y, y_pred)
        rmse = np.sqrt(mean_squared_error(y, y_prob))

        # Store results
        for col, coef in zip(
            X.columns[1:], result.params[1:]
        ):  # Skip intercept for Variable names
            results_data_statsmodels.append(
                {
                    "Group": group,
                    "Variable": col,
                    "Coefficient": coef,
                    "Intercept": result.params["const"],
                    "Log-Likelihood": log_likelihood,
                    "AIC": result.aic,
                    "BIC": result.bic,
                    "Precision": precision,
                    "Recall": recall,
                    "F1-Score": f1,
                    "Accuracy": accuracy,
                    "RMSE": rmse,
                }
            )
    except Exception as e:
        print(f"Error processing group {group}: {e}")

# Create DataFrame with results
statsmodels_results_df = pd.DataFrame(results_data_statsmodels)


c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)
c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)
c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)
c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)
c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg

Error processing group 00_18_F: Singular matrix


c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)
c:\Users\huber\anaconda3\envs\python_r_env\Lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)


In [5]:
statsmodels_results_df

,Group,Variable,Coefficient,Intercept,Log-Likelihood,AIC,BIC,Precision,Recall,F1-Score,Accuracy,RMSE
0,00_18_M,2,0.067884,-1.479076,-1431.978618,2873.957235,2903.835059,0.739130,0.428752,0.542698,0.803025,0.396131
1,00_18_M,3,-0.222142,-1.479076,-1431.978618,2873.957235,2903.835059,0.739130,0.428752,0.542698,0.803025,0.396131
2,00_18_M,4,0.243263,-1.479076,-1431.978618,2873.957235,2903.835059,0.739130,0.428752,0.542698,0.803025,0.396131
3,00_18_M,5,2.520530,-1.479076,-1431.978618,2873.957235,2903.835059,0.739130,0.428752,0.542698,0.803025,0.396131
4,19_34_F,2,-0.021280,-1.847588,-3793.393741,7596.787481,7632.076922,0.000000,0.000000,0.000000,0.833100,0.369191
5,19_34_F,3,0.190548,-1.847588,-3793.393741,7596.787481,7632.076922,0.000000,0.000000,0.000000,0.833100,0.369191
6,19_34_F,4,0.835987,-1.847588,-3793.393741,7596.787481,7632.076922,0.000000,0.000000,0.000000,0.833100,0.369191
7,19_34_F,5,-0.104617,-1.847588,-3793.393741,7596.787481,7632.076922,0.000000,0.000000,0.000000,0.833100,0.369191
8,19_34_M,2,-2.053228,0.676482,-16797.086603,33604.173207,33645.104683,0.662953,0.335631,0.445646,0.648528,0.470394
9,19_34_M,3,-0.949023,0.676482,-16797.086603,33604.173207,33645.104683,0.662953,0.335631,0.445646,0.648528,0.470394
